In [13]:
from dotenv import load_dotenv
load_dotenv("../agentic_rag_homework/.env")

True

In [19]:
from openai import OpenAI
import os

# Initialize client using the variables injected from your .env
openai_client = OpenAI(
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN")
)

print(os.getenv("BASE_URL"))

https://models.inference.ai.azure.com


In [15]:
from ingest import load_faq_data
documents = load_faq_data()
len(documents)

72

In [20]:

from rag_agent import SearchAgent
from ingest import build_chunks, build_index

chunks = build_chunks(documents)
index = build_index(chunks)


agent = SearchAgent(index=index, openai_client=openai_client)
result = agent.agent_loop(
    question="How does the agentic loop work, and how is it different from plain RAG?",
    model="gpt-4o-mini"
)

iteration #1...
function_call: search {"query":"agentic loop vs RAG"}
iteration #2...
ASSISTANT:
The concepts of the "agentic loop" and RAG (Retrieval-Augmented Generation) are related but different, as they address distinct challenges in building intelligent systems. Here's a breakdown of each concept and how they differ:

---

### **Agentic Loop**
- The agentic loop refers to a design pattern where an AI agent controls the flow of decision-making, select actions dynamically, and interacts with different tools or sources as needed to achieve its goal. It is iterative and involves multiple steps in which the system can:
  1. Analyze a task.
  2. Decide if it has enough information to act or if it needs to call external tools, perform another round of retrieval, or seek clarification from the user.
  3. Refine its understanding, execute the action, and repeat until the task is completed.

- **How it works:**
  1. The agent receives a user prompt.
  2. It consults its knowledge base and 

In [21]:
print("Q1 : "+str(len(documents)))

Q1 : 72


In [23]:
doc_index = build_index(documents)
results = doc_index.search("How does the agentic loop keep calling the model until it stops?")
print(results[0]["filename"])

01-agentic-rag/lessons/14-agentic-loop.md


In [24]:
query = "How does the agentic loop keep calling the model until it stops?"

# search
search_results = doc_index.search(query, num_results=5)

# build context
context = ""
for doc in search_results:
    context += doc["filename"] + "\n"
    context += doc["content"] + "\n\n"

# build prompt
prompt = f"""Question: {query}

Context:
{context}"""

# call LLM once
messages = [
    {"role": "system", "content": "You are a course teaching assistant. Answer based on the context provided."},
    {"role": "user", "content": prompt}
]

response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
)

print("Q3 input tokens:", response.usage.prompt_tokens)
print(response.choices[0].message.content)

Q3 input tokens: 7077
The agentic loop functions by continuously calling the model in a structured manner until the model determines it has enough information to provide a final answer. Here's how it works:

1. **Initialization**: The loop starts with messages that include instructions given to the model (in the role of an agent) and a user question. This sets the context for the agent's responses.

2. **API Call**: Inside a `while` loop, the agent makes an API call to the model with the current conversation history (messages). 

3. **Response Processing**:
   - The model processes the input and can decide whether it needs to search for more information.
   - If it determines it requires further information, it generates a function call (e.g., to `search`) which is included in the response.

4. **Function Call Handling**: 
   - If the response from the model contains a `function_call`, the helper function `make_call()` executes that function (like a search) and captures the result. Thi

In [25]:
print(len(chunks))

295


In [26]:
# same query but on chunk index
search_results_chunks = index.search(query, num_results=5)

context_chunks = ""
for doc in search_results_chunks:
    context_chunks += doc["filename"] + "\n"
    context_chunks += doc["content"] + "\n\n"

prompt_chunks = f"""Question: {query}

Context:
{context_chunks}"""

messages_chunks = [
    {"role": "system", "content": "You are a course teaching assistant. Answer based on the context provided."},
    {"role": "user", "content": prompt_chunks}
]

response_chunks = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages_chunks
)

print("Q5 input tokens:", response_chunks.usage.prompt_tokens)
print("Q3 tokens:", response.usage.prompt_tokens)
print("Ratio Q3/Q5:", round(response.usage.prompt_tokens / response_chunks.usage.prompt_tokens))

Q5 input tokens: 2260
Q3 tokens: 7077
Ratio Q3/Q5: 3
